Note that the following paths (relative to the directory in which the code is run) must exist for the code to run correctly.  Data is assumed to be in these directories:
- "../Data/train"
    - This directory must contain the training data with files within their respective genre folder
- "../Data/test"
    - This directory must contain the test data
- "../Output/"
    - Output files will be written to this directory
- "../Data_Proc/train"
    - Training data will be processed into spectrogram images and saved here
- "../Data_Proc/test"
    - Test data will be processed into spectrogram images and saved here
- "../Eval_Data/train"
    - Training spectrograms will be split into train and test sets. The train split will be put here
- "../Eval_Data/test"
    - Training spectrograms will be split into train and test sets. The test split will be put here

First, load the necessary packages:

In [ ]:
# Import Necessary Packages
import os
import librosa
import numpy as np
import pandas as pd
import seaborn as sn
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import pygad
from IPython.display import Audio, display
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from Data_Load_and_Process import (proc_all_au_data, proc_all_au_test_data, 
        get_all_au_test_data, get_all_au_data, get_all_au_data_labels, 
        split_spec_data_for_CNN_test_eval, create_class_dirs_in_eval_dirs,
        train_test_split_CNN_data, move_eval_files, proc_all_au_data_rgb, proc_all_au_test_data_rgb)

Load necessary pytorch packages:

In [ ]:
import torch
#import torch.nn as nn
#import torch.optim as optim
from torch import nn, optim
from torchinfo import summary
from torchmetrics import Accuracy
from torchvision import transforms, datasets
import pytorch_lightning as pl
from torch.utils.data import Dataset
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import DataLoader, TensorDataset, random_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning import LightningModule, LightningDataModule, Trainer, seed_everything

Initialize necessary variables (including paths):

In [ ]:
# Features to extract from raw training data:
selected_features = ["mfcc", "chroma", "spectral contrast", "zero crossings"]
selected_features = ["stft"]
selected_features = ["mfcc"]
selected_features = ["melspec"]

# Data input and output directories:
raw_data_path = "../Data"
train_data_path = "../Data/train"
test_data_path = "../Data/test"
sample_input_path = "../Data/sandbox"
debug_data_directory = sample_input_path
proc_data_path = "../Data_Proc"
proc_train_data_path = "../Data_Proc/train"
proc_test_data_path = "../Data_Proc/test"
eval_data_path = "../Eval_Data"
eval_train_data_path = "../Eval_Data/train"
eval_test_data_path = "../Eval_Data/test"

Load data from audio files, extract features, stack together and save as images in a processed data directory ("../Data_Proc").  Then split training data into train and test sets for evaluation ("../Eval_Data").  This data will be used in CNN evaluation and transfer learning evaluation below.

In [ ]:
# Load, feature extract, save as image:
proc_all_au_data(selected_features)
proc_all_au_test_data(selected_features)

# Get data labels
Y, encoder = get_all_au_data_labels()


# Get list of paths and processed data (for CNN)
class_list, path_list = split_spec_data_for_CNN_test_eval()

# Create class directories in evaluation directories:
create_class_dirs_in_eval_dirs(class_list)

# Split evaluation data lists into training and test lists
trainpath_list, testpath_list, trainclass_list, testclass_list = train_test_split_CNN_data(class_list, path_list)

# Move processed train files to evaluation train and test directories
move_eval_files(trainpath_list, trainclass_list, eval_train_data_path)
move_eval_files(testpath_list, testclass_list, eval_test_data_path)

### CNN Section:

Define necessary classes:
1. First, since datasets.ImageFolder() requires a directory structure where data is organized into classes, we cannot use it to load the test data.  We therefore define ClasslessImageFolder() to use for loading the test data
2. We define a class for loading and transforming the "audio" spectrogram image data: AudioDataModule() which inherits from LightningDataModule

In [ ]:

class ClasslessImageFolder(Dataset):
    def __init__(self, root, transform=None):
        self.file_paths = sorted(list(Path(root).glob("*.png")))
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        image_data = Image.open(file_path).convert("RGB")
        if self.transform:
            image_data = self.transform(image_data)

        return image_data, file_path.name


class AudioDataModule(LightningDataModule):
    def __init__(self, data_dir=eval_data_path, predict_data_dir=proc_test_data_path, batch_size=64, num_workers=32):
        super().__init__()
        self.data_dir = data_dir
        self.predict_data_dir = predict_data_dir
        self.train_data_dir = os.path.join(data_dir, "train")
        self.test_data_dir = os.path.join(data_dir, "test")
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.transform_train = transforms.Compose([
            transforms.ToTensor()
        ])

        self.transforms_test = transforms.Compose([
            transforms.ToTensor()
        ])

    def prepare_data(self) -> None:
        pass

    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            full_dataset = datasets.ImageFolder(root=self.train_data_dir, 
                                                transform=self.transform_train)
            train_size = int(0.9 * len(full_dataset))
            val_size = len(full_dataset) - train_size
            self.train_dataset, self.val_dataset = random_split(full_dataset, 
                                                                [train_size, val_size])
            self.classnames = full_dataset.classes
            self.class_to_idx = full_dataset.class_to_idx
        elif stage == 'test':
            self.test_dataset = datasets.ImageFolder(root=self.test_data_dir, 
                                                     transform=self.transforms_test)
        elif stage == 'predict':
            self.predict_dataset = ClasslessImageFolder(root=self.predict_data_dir, 
                                                     transform=self.transforms_test)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size,
                          shuffle=True, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers)
    
    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers)
    
    def predict_dataloader(self):
        return DataLoader(self.predict_dataset, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers)





### Class to dynamically construct CNN

In [ ]:
class CNNModelDynamic(LightningModule):
    def __init__(self, lr=1e-03, num_classes=10, dropout_rate=0.4, base_filters=32, num_convolution_blocks=4, kernel_size = (2,4)):
        super().__init__()
        self.save_hyperparameters()
        
        if self.hparams.num_convolution_blocks == 1:
            self.model = nn.Sequential(
                # Block 1: Initial feature extraction
                nn.Conv2d(in_channels=3, out_channels=self.hparams.base_filters, kernel_size=3, padding=1),
                nn.BatchNorm2d(base_filters),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=self.hparams.kernel_size), 

                nn.AdaptiveAvgPool2d((1, 1)), 
                nn.Flatten(),

                # Classification Head
                nn.Dropout(self.hparams.dropout_rate),
                nn.Linear(base_filters, 256),
                nn.ReLU(),
                nn.Dropout(self.hparams.dropout_rate),
                nn.Linear(256, self.hparams.num_classes)
            )

            self.acc = Accuracy(task="multiclass", num_classes=self.hparams.num_classes)
        
        if self.hparams.num_convolution_blocks == 2:
            self.model = nn.Sequential(
                # Block 1: Initial feature extraction
                nn.Conv2d(in_channels=3, out_channels=self.hparams.base_filters, kernel_size=3, padding=1),
                nn.BatchNorm2d(base_filters),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=self.hparams.kernel_size), 

                # Block 2
                nn.Conv2d(in_channels=self.hparams.base_filters, out_channels=self.hparams.base_filters * 2, kernel_size=3, padding=1),
                nn.BatchNorm2d(self.hparams.base_filters * 2),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=self.hparams.kernel_size), 

                nn.AdaptiveAvgPool2d((1, 1)), 
                nn.Flatten(),

                # Classification Head
                nn.Dropout(self.hparams.dropout_rate),
                nn.Linear(self.hparams.base_filters * 2, 256),
                nn.ReLU(),
                nn.Dropout(self.hparams.dropout_rate),
                nn.Linear(256, self.hparams.num_classes)
            )

            self.acc = Accuracy(task="multiclass", num_classes=self.hparams.num_classes)
        
        if self.hparams.num_convolution_blocks == 3:
            self.model = nn.Sequential(
                # Block 1: Initial feature extraction
                nn.Conv2d(in_channels=3, out_channels=self.hparams.base_filters, kernel_size=3, padding=1),
                nn.BatchNorm2d(base_filters),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=self.hparams.kernel_size), 

                # Block 2
                nn.Conv2d(in_channels=self.hparams.base_filters, out_channels=self.hparams.base_filters * 2, kernel_size=3, padding=1),
                nn.BatchNorm2d(self.hparams.base_filters * 2),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=self.hparams.kernel_size), 

                # Block 3
                nn.Conv2d(in_channels=self.hparams.base_filters * 2, out_channels=self.hparams.base_filters * 3, kernel_size=3, padding=1),
                nn.BatchNorm2d(self.hparams.base_filters * 3),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=(2, 2)), # Result: ~16 x 37

                nn.AdaptiveAvgPool2d((1, 1)), 
                nn.Flatten(),

                # Classification Head
                nn.Dropout(self.hparams.dropout_rate),
                nn.Linear(self.hparams.base_filters * 3, 256),
                nn.ReLU(),
                nn.Dropout(self.hparams.dropout_rate),
                nn.Linear(256, self.hparams.num_classes)
            )

            self.acc = Accuracy(task="multiclass", num_classes=self.hparams.num_classes)

            
        if self.hparams.num_convolution_blocks == 4:
            self.model = nn.Sequential(
                # Block 1: Initial feature extraction
                nn.Conv2d(in_channels=3, out_channels=self.hparams.base_filters, kernel_size=3, padding=1),
                nn.BatchNorm2d(base_filters),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=self.hparams.kernel_size), 

                # Block 2
                nn.Conv2d(in_channels=self.hparams.base_filters, out_channels=self.hparams.base_filters * 2, kernel_size=3, padding=1),
                nn.BatchNorm2d(self.hparams.base_filters * 2),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=self.hparams.kernel_size), 

                # Block 3
                nn.Conv2d(in_channels=self.hparams.base_filters * 2, out_channels=self.hparams.base_filters * 3, kernel_size=3, padding=1),
                nn.BatchNorm2d(self.hparams.base_filters * 3),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=(2, 2)), 

                # Block 4
                nn.Conv2d(in_channels=self.hparams.base_filters * 3, out_channels=self.hparams.base_filters * 4, kernel_size=3, padding=1),
                nn.BatchNorm2d(self.hparams.base_filters * 4),
                nn.ReLU(),

                nn.AdaptiveAvgPool2d((1, 1)), 
                nn.Flatten(),

                # Classification Head
                nn.Dropout(self.hparams.dropout_rate),
                nn.Linear(self.hparams.base_filters * 4, 256),
                nn.ReLU(),
                nn.Dropout(self.hparams.dropout_rate),
                nn.Linear(256, self.hparams.num_classes)
            )

            self.acc = Accuracy(task="multiclass", num_classes=self.hparams.num_classes)            
            
        
    def forward(self, x):
        return self.model(x)
    
    def shared_step(self, batch, stage=None):
        x, y = batch
        logits = self(x)
        loss = nn.CrossEntropyLoss()(logits, y)
        preds = logits.argmax(1)
        acc = self.acc(preds, y)
        
        if stage:
            self.log(f"{stage}_loss", loss, on_epoch=True, prog_bar=True, logger=True)
            self.log(f"{stage}_acc", acc, on_epoch=True, prog_bar=True, logger=True)
        return loss, acc
    
    def training_step(self, batch, batch_idx):
        loss, _ = self.shared_step(batch, "train")
        return loss
    
    def validation_step(self, batch, batch_idx):
        loss, _ = self.shared_step(batch, "val")
        return loss
    
    def test_step(self, batch, batch_idx):
        loss, _ = self.shared_step(batch, "test")
        return loss
    
    def predict_step(self, batch, batch_idx):
        x, filenames = batch
        logits = self(x)
        return logits, filenames
    
    def configure_optimizers(self):
        # Using Adam as suggested for adaptive learning rates
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        # Scheduler reduces LR when validation loss plateaus
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=3, factor=0.5, mode='min'
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss"
            }
        }

### Random Tests

In [ ]:

# Number of classes
Y, encoder = get_all_au_data_labels()
num_classes = len(encoder.classes_)

# Summarize model on a dummy model
#dummy = torch.zeros(1, 3, 32, 1200)
#print(dummy.shape)
#model_visual = CNNModel()
#summary(model_visual, input_data=dummy)

# Set up data module and model for testing/validation
dm = AudioDataModule(batch_size=64, num_workers=32)
#model = CNNModel(lr=1e-3, num_classes=num_classes)
model = CNNModelDynamic(lr=1e-03, num_classes=10, dropout_rate=0.4, base_filters=32, num_convolution_blocks=4, kernel_size = (2,4))
# Checkpoint settings
checkpoint_cb = ModelCheckpoint(
    monitor="val_loss", save_top_k=1, mode="min", filename="cnn-scratch"#,
    #save_on_train_epoch_end=False
)

# Initialize trainer
trainer = Trainer(
    max_epochs=5,
    accelerator="auto",
    logger=CSVLogger(save_dir="./logs/"),
    devices="auto",
    #callbacks=[checkpoint_cb, EarlyStopping(monitor="val_loss", patience=3),
               #LearningRateMonitor(logging_interval="epoch")],
    #enable_checkpointing = True,
    enable_model_summary = True,
)

# Fit training and test data
trainer.fit(model, dm)
test_results = trainer.test(model, dm)
print(test_results)

# Make Predictions
test_predictions = trainer.predict(model, dm)

# Plot training data
metrics = pd.read_csv(f"{trainer.logger.log_dir}/metrics.csv")
del metrics["step"]
metrics.set_index("epoch", inplace=True)
sn.relplot(data=metrics[['train_acc_epoch', 'val_acc', 
                       'train_loss_epoch', 'val_loss']], kind="line")

## Fitness Function and "On Generation" Function

In [ ]:
# Load data
dm = AudioDataModule(batch_size=64, num_workers=32)

# Search space
learning_rates = [1e-3, 0.5e-3, 1e-4, 0.5e-4, 1e-5]
kernels = [(1,1),(1,2),(1,3),(1,4),(1,5),(1,6),(1,7)]
base_filters = [16, 32, 64]
num_convolution_blocks = [1, 2, 3, 4]
dropout_ratio = [0.2, 0.3, 0.4, 0.5, 0.6]

gene_space = [
    range(len(learning_rates)),
    range(len(kernels)),
    range(len(base_filters)),
    range(len(num_convolution_blocks)),
    range(len(dropout_ratio))
]


# Fitness function that is passed to PyGad
def fitness_func(ga_instance, solution, solution_idx):
    
    # Decode genes into parameters for the CNN
    learn_rate = learning_rates[int(solution[0])]
    pool_k = kernels[int(solution[1])]
    filters = base_filters[int(solution[2])]
    num_blocks = num_convolution_blocks[int(solution[3])]
    dropout = dropout_ratio[int(solution[4])]
    
    # Instantiate model with gene parameters
    model = CNNModelDynamic(lr=learn_rate,
                                 num_classes=10,
                                 dropout_rate=dropout,
                                 base_filters=filters,
                                 num_convolution_blocks=num_blocks,
                                 kernel_size = pool_k)
        
    # If running across multiple GPUs
    #process_id = os.getpid()
    #available_gpus = [0]
    #gpu_id = available_gpus[process_id % len(available_gpus)]
    #gpu_id = available_gpus[(process_id % 10)] #Try two parallel processes on single gpu
    
    # If running on single GPU
    available_gpus = [0]
    gpu_id = available_gpus[0]
    
    # Setup trainer
    trainer = Trainer(
        max_epochs=20, 
        accelerator="gpu",
        devices=[gpu_id],
        logger=False, 
        enable_checkpointing=False
    )
    
    # Train and validate
    trainer.fit(model, dm)
    metrics = trainer.validate(model, dm)
    
    # Extract accuracy as fitness
    accuracy = metrics[0]['val_acc'] 
    return accuracy


# List to store history: 
ga_history = []

def on_generation(ga_instance):
    generation = ga_instance.generations_completed
    
    # Calculate ratio of unique binary strings in each generation
    population = ga_instance.population
    unique_strands = np.unique(population, axis=0)
    diversity_ratio = len(unique_strands) / len(population)
    
    # Get all chromosomes (binary strings) and their fitness scores
    population_dna = ga_instance.population.copy()
    fitness_scores = ga_instance.last_generation_fitness.copy()
    
    # Store the data
    ga_history.append({
        'generation': generation,
        'chromosomes': population_dna,
        'fitness': fitness_scores,
        'best_fitness': np.max(fitness_scores),
        'diversity_ratio': diversity_ratio
    })
    
    print(f"Generation {generation} saved. Best Fitness: {np.max(fitness_scores)}")


## Run the GA

In [ ]:

# Initialize GA with PyGAD

ga_instance = pygad.GA(
    on_generation = on_generation,
    parallel_processing = ["process",5], # Launch parallel processes
    num_generations=50,
    num_parents_mating=5,
    fitness_func=fitness_func,
    sol_per_pop=10,
    num_genes=len(gene_space),
    gene_space=gene_space,
    mutation_percent_genes=20,
    save_best_solutions=True,
    save_solutions = True
)



# Start the evolutionary process
ga_instance.run()

# Print the best bit string:
solution, solution_fitness, solution_idx = ga_instance.best_solution()
print(f"Best Architecture String: {solution}")

In [ ]:
# To extend by another X generations:
ga_instance.num_generations = 10 # Set the number of *additional* generations
ga_instance.run()

## Post-processing and analysis

In [ ]:
# Print best values for each gene

solution, solution_fitness, solution_idx = ga_instance.best_solution()

lr = learning_rates[int(solution[0])]
pool_k = kernels[int(solution[1])]
filters = base_filters[int(solution[2])]
num_blocks = num_convolution_blocks[int(solution[3])]
dropout = dropout_ratio[int(solution[4])]

print(f"Best Learning Rate: {lr}")
print(f"Best Max Pooling Size: {pool_k}")
print(f"Best Base Filter Size: {filters}")
print(f"Best Num. Convolution Blocks: {num_blocks}")
print(f"Best Dropout Ratio: {dropout}")


In [ ]:
#Plot fitness vs. generation
ga_instance.plot_fitness()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract all best chromosomes from history
best_dna_history = [entry['chromosomes'][np.argmax(entry['fitness'])] for entry in ga_history]

# Heat map showing best string vs. generation
plt.figure(figsize=(10, 6))
sns.heatmap(best_dna_history, cmap='YlGnBu')
plt.xlabel("Gene", fontsize=14)
plt.ylabel("Generation", fontsize=14)
plt.title("Evolution of Best Network DNA", fontsize=16)
plt.show()

# Plot diveristy ratio of each generation
generation = [entry['generation'] for entry in ga_history]
diversity_ratio = [entry['diversity_ratio'] for entry in ga_history]

plt.figure(figsize=(10, 6))
plt.xlabel('Generation', fontsize = 14)
plt.ylabel('Diversity Ratio', fontsize = 14)
plt.title('Diversity Ratio vs. Generation', fontsize = 16)
plt.plot(generation, diversity_ratio, marker='o',linewidth=3)
#plt.xticks(generation, fontsize=12)
plt.yticks(fontsize=12)

plt.grid(True)

In [ ]:

plot_data = []
for entry in ga_history:
    for f in entry['fitness']:
        plot_data.append({'Generation': entry['generation'], 'Fitness': f})

df = pd.DataFrame(plot_data)

plt.figure(figsize=(12, 6))
sns.boxplot(x='Generation', y='Fitness', data=df, palette='viridis')
plt.title('Fitness Distribution vs. Generation', fontsize=20)
plt.xlabel('Generation', fontsize=16)
plt.ylabel('Fitness (Classification Accuracy)', fontsize=16)
plt.show()



In [ ]:
# Convert the list of solutions into a NumPy array
all_evaluations = np.array(ga_instance.solutions)

# Find unique rows (individuals)
# axis=0 treats each chromosome as a single unit
unique_individuals = np.unique(all_evaluations, axis=0)

total_unique_count = len(unique_individuals)
total_evaluations = len(all_evaluations)

print(f"Total individuals processed: {total_evaluations}")
print(f"Total unique individuals: {total_unique_count}")
print(f"Redundancy rate: {100 * (1 - total_unique_count/total_evaluations):.2f}%")

In [ ]:
data = []
for entry in ga_history:
    # Use the key 'chromosomes' as defined in your on_generation function
    for i, solution in enumerate(entry['chromosomes']): 
        data.append({
            'Generation': entry['generation'],
            'Learning Rate': solution[0],
            'Kernel': solution[1],
            'Base Filters': solution[2],
            'Blocks': solution[3],
            'Dropout': solution[4],
            'Fitness': entry['fitness'][i]
        })

df = pd.DataFrame(data)

In [ ]:
# Heat maps comparing accuracy of blocks and other features
pivot_table_blocks_filter = df.pivot_table(values='Fitness', 
                             index='Blocks', 
                             columns='Base Filters', 
                             aggfunc='mean')

pivot_table_blocks_kernel = df.pivot_table(values='Fitness', 
                             index='Blocks', 
                             columns='Kernel', 
                             aggfunc='mean')

pivot_table_blocks_kernel = df.pivot_table(values='Fitness', 
                             index='Blocks', 
                             columns='Kernel', 
                             aggfunc='mean')

pivot_table_blocks_lr = df.pivot_table(values='Fitness', 
                             index='Blocks', 
                             columns='Learning Rate', 
                             aggfunc='mean')

pivot_table_blocks_dr = df.pivot_table(values='Fitness', 
                             index='Blocks', 
                             columns='Dropout', 
                             aggfunc='mean')

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_table_blocks_filter, annot=True, cmap='YlGnBu')
plt.title('Mean Fitness: Blocks vs. Base Filters', fontsize=16)
plt.xlabel('Base Filters', fontsize=12)
plt.ylabel('Number of Convolution Blocks', fontsize=12)
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_table_blocks_kernel, annot=True, cmap='YlGnBu')
plt.title('Mean Fitness: Blocks vs. Kernel Size', fontsize=16)
plt.xlabel('Kernel Size', fontsize=12)
plt.ylabel('Number of Convolution Blocks', fontsize=12)
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_table_blocks_lr, annot=True, cmap='YlGnBu')
plt.title('Mean Fitness: Blocks vs. Learning Rate', fontsize=16)
plt.xlabel('Learning Rate', fontsize=12)
plt.ylabel('Number of Convolution Blocks', fontsize=12)
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_table_blocks_dr, annot=True, cmap='YlGnBu')
plt.title('Mean Fitness: Blocks vs. Dropout Rate', fontsize=16)
plt.xlabel('Dropout Rate', fontsize=12)
plt.ylabel('Number of Convolution Blocks', fontsize=12)
plt.show()

In [ ]:
# Heat maps comparing accuracy of blocks and other features
pivot_table_filter_blocks = df.pivot_table(values='Fitness', 
                             index='Base Filters', 
                             columns='Blocks', 
                             aggfunc='mean')

pivot_table_filter_kernel = df.pivot_table(values='Fitness', 
                             index='Base Filters', 
                             columns='Kernel', 
                             aggfunc='mean')

pivot_table_filter_kernel = df.pivot_table(values='Fitness', 
                             index='Base Filters', 
                             columns='Kernel', 
                             aggfunc='mean')

pivot_table_filter_lr = df.pivot_table(values='Fitness', 
                             index='Base Filters', 
                             columns='Learning Rate', 
                             aggfunc='mean')

pivot_table_filter_dr = df.pivot_table(values='Fitness', 
                             index='Base Filters', 
                             columns='Dropout', 
                             aggfunc='mean')


plt.figure(figsize=(10, 6))
sns.heatmap(pivot_table_filter_blocks, annot=True, cmap='YlGnBu')
plt.title('Mean Fitness: Base Filters vs. Blocks', fontsize=16)
plt.xlabel('Number of Convolution Blocks', fontsize=12)
plt.ylabel('Base Filters', fontsize=12)
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_table_filter_kernel, annot=True, cmap='YlGnBu')
plt.title('Mean Fitness: Base Filters vs. Kernel Size', fontsize=16)
plt.xlabel('Kernel Size', fontsize=12)
plt.ylabel('Base Filters', fontsize=12)
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_table_filter_lr, annot=True, cmap='YlGnBu')
plt.title('Mean Fitness: Base Filters vs. Learning Rate', fontsize=16)
plt.xlabel('Learning Rate', fontsize=12)
plt.ylabel('Base Filters', fontsize=12)
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_table_filter_dr, annot=True, cmap='YlGnBu')
plt.title('Mean Fitness: Base Filters vs. Dropout Rate', fontsize=16)
plt.xlabel('Dropout Rate', fontsize=12)
plt.ylabel('Base Filters', fontsize=12)
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
# Compute correlation between genes and fitness
corr_matrix = df.drop(columns=['Generation']).corr(method='spearman')
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Gene-Fitness Correlation Matrix', fontsize=18)
plt.show()

In [ ]:
# Save GA instance
#ga_instance.save(filename="Batch6")